# OpenVLA: LIBERO-Object Evaluation (Baseline vs MMD-Selective)

This notebook evaluates our two trained LoRA checkpoints on the LIBERO-Object task suite. 
It is configured to run headless in the Kaggle GPU environment.

## 1. System & Dependencies Setup

In [ ]:
# 1. Install EGL headless rendering dependencies for MuJoCo/robosuite
!apt-get update && apt-get install -y libegl1-mesa libegl1-mesa-dev libgl1-mesa-glx libosmesa6

# 2. Clone Repositories
!git clone https://github.com/openvla/openvla.git
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git

# 3. Install OpenVLA and LIBERO Python dependencies
!cd openvla && pip install -e .
!cd openvla/experiments/robot/libero && pip install -r libero_requirements.txt

# Force robosuite 1.4.1 (which uses DeepMind mujoco bindings)
!pip install robosuite==1.4.1 mujoco easydict cloudpickle gym bddl imageio[ffmpeg]

## 2. Environment Variables & Patching
We must export EGL variables for headless rendering and patch `openvla_utils.py` for T4 compatibility.

In [ ]:
import os

# Force hardware-accelerated headless rendering
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["MUJOCO_GL"] = "egl"

# Patch 1: SDPA attention (fixes Ampere-only crash)
!sed -i 's/attn_implementation="flash_attention_2"/attn_implementation="sdpa"/' /kaggle/working/openvla/experiments/robot/openvla_utils.py

# Patch 2: bfloat16 to float16 (fixes T4 missing native bfloat16)
!sed -i 's/bfloat16/float16/g' /kaggle/working/openvla/experiments/robot/openvla_utils.py
!sed -i 's/bfloat16/float16/g' /kaggle/working/openvla/experiments/robot/libero/run_libero_eval.py

!echo "✅ Patched openvla source code for T4 compatibility (sdpa, float16)"

## 3. Extract Datasets & Check Paths
We need to verify the exact path to the adapter checkpoint directories because OpenVLA creates nested folders.

In [ ]:
!pip install -e LIBERO

# Check Kaggle Input directory structure
# Look for the folder that contains `adapter_model.safetensors`
!ls -la /kaggle/input/

## 4. Run Evaluation (Baseline Uniform LoRA)
**Dry Run**: `--num_trials_per_task 1` is set below to verify EGL rendering and adapter merging without wasting hours. Change this to 15 once verified. 

Note the addition of `--load_in_4bit True` to prevent VRAM OOM on the 16GB T4.

In [ ]:
import sys
import os

# Set LIBERO python path
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:/kaggle/working/LIBERO"

# IMPORTANT: Update this path to the EXACT directory containing adapter_model.safetensors
baseline_checkpoint = "/kaggle/input/openvla-baseline-uniform-lora-run/openvla_baseline_uniform_lora_run/checkpoint-2000"

print("Starting Baseline Evaluation...")
!cd openvla && python experiments/robot/libero/run_libero_eval.py \
    --model_family openvla \
    --pretrained_checkpoint {baseline_checkpoint} \
    --task_suite_name libero_object \
    --center_crop True \
    --load_in_4bit True \
    --num_trials_per_task 1

## 5. Run Evaluation (MMD-Selective LoRA)

In [ ]:
# IMPORTANT: Update this path to the EXACT directory containing adapter_model.safetensors
mmd_checkpoint = "/kaggle/input/mmd-selective-lora-run/mmd_selective_lora_run/checkpoint-2000"

print("Starting MMD-Selective Evaluation...")
!cd openvla && python experiments/robot/libero/run_libero_eval.py \
    --model_family openvla \
    --pretrained_checkpoint {mmd_checkpoint} \
    --task_suite_name libero_object \
    --center_crop True \
    --load_in_4bit True \
    --num_trials_per_task 1